In [1]:
import duckdb
import pandas as pd
import numpy as np
#%sql duckdb:///path/to/file.db # file-based database
# no trailing comments after %sql or you break the magic
%load_ext sql
%sql duckdb:///:memory:

np.random.seed(22)
suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMg", "EastCoast"]
categories = ["Electronics", "Hardware", "Consumables", "Apparel"]
warehouses = ["East", "West", "Central"]

rows = []
for i in range(1, 201):
    rows.append({
        "po_id":          f"PO-{str(i).zfill(4)}",
        "supplier":       np.random.choice(suppliers),
        "category":       np.random.choice(categories),
        "warehouse":      np.random.choice(warehouses),
        "order_date":     pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(np.random.randint(0, 365))),
        "amount":         round(np.random.uniform(500, 50000), 2),
        "units":          np.random.randint(10, 500),
        "lead_time_days": np.random.randint(3, 45),
        "on_time":        np.random.choice([True, False], p=[0.8, 0.2])
    })

po = pd.DataFrame(rows)

supplier_master = pd.DataFrame({
     "supplier": ["Acme", "GlobalCo", "FastParts", "PrimeMfg"],
    "country":  ["USA", "UK", "Germany", "France"],
    "rating":   [4.5, 3.8, 4.2, 4.0],
    "approved": [True, True, True, False]
})

%sql --persist po
%sql --persist supplier_master
print(f"po: {po.shape}, supplier_master: {supplier_master.shape}")

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

Success! Persisted po to the database.

Running query in 'duckdb:///:memory:'

Success! Persisted supplier_master to the database.

po: (200, 9), supplier_master: (4, 4)


In [ ]:
%%sql

-- comment beneath the %%sql magic
-- 2.1 -- INNER JOIN
-- inner joins return only those rows that match in both tables
-- rows with no match are dropped -- eg PrimMg not matching PrimeMfg
-- Joing po to supplier_master on supplier
-- return po_id, supplier, amount, country, rating

select
    p.po_id,
    p.supplier,
    p.amount,
    s.country,
    s.rating
from
    po p
inner join
    supplier_master s
on
    p.supplier = s.supplier
order by
    p.amount desc
limit 10


    

Running query in 'duckdb:///:memory:'

po_id,supplier,amount,country,rating
PO-0083,Acme,49123.34,USA,4.5
PO-0045,GlobalCo,47711.66,UK,3.8
PO-0050,FastParts,47326.18,Germany,4.2
PO-0004,GlobalCo,47315.6,UK,3.8
PO-0030,GlobalCo,47185.62,UK,3.8
PO-0126,GlobalCo,47108.24,UK,3.8
PO-0196,FastParts,47054.5,Germany,4.2
PO-0137,Acme,46565.15,USA,4.5
PO-0142,FastParts,43990.25,Germany,4.2
PO-0082,FastParts,43382.41,Germany,4.2


In [ ]:
%%sql

-- S2.2 -- LEFT JOIN
-- All rows from the left table matching the right, NaN where no match is possible
-- keep all POs even if supplier has no master record
-- left table is supplier_master
-- return: po_id, supplier, amount, country
-- then filter for rows where country IS NULL
-- these are the unregistered suppliers

select
    p.po_id,
    p.supplier,
    p.amount,
    s.country
from
    po p
left join
    supplier_master s
on
    p.supplier = s.supplier
where
    s.country is null
order by
    p.amount desc



In [6]:
po.head(5)

,po_id,supplier,category,warehouse,order_date,amount,units,lead_time_days,on_time
0,PO-0001,EastCoast,Electronics,East,2024-12-22,32055.22,94,11,True
1,PO-0002,FastParts,Consumables,East,2024-02-15,23470.66,103,42,True
2,PO-0003,Acme,Electronics,West,2024-01-28,22382.64,39,21,True
3,PO-0004,GlobalCo,Apparel,Central,2024-01-08,47315.60,33,26,True
4,PO-0005,EastCoast,Apparel,East,2024-12-25,37347.63,123,4,False


In [7]:
supplier_master.head(3)

,supplier,country,rating,approved
0,Acme,USA,4.5,True
1,GlobalCo,UK,3.8,True
2,FastParts,Germany,4.2,True


In [ ]:
%%sql

-- S2.3 JOIN WITH AGGREGATION
-- LEFT table: po (p)
-- RIGHT table: supplier_master (s)
-- join type: keeps only matched rows
-- group by country
-- show: country, po_count, total_spend, avg_rating
-- filter: only approved suppliers
-- sort: total_spend descending

select
    s.country,
    count(*) as po_count,                           -- counts every row in the group
    ROUND(sum(p.amount), 2) as total_spend,
    ROUND(avg(s.rating), 2) as avg_rating
from
    po p
inner join
    supplier_master s
on
    p.supplier = s.supplier
where
    s.approved = true
group by
    s.country
order by
    total_spend desc





Running query in 'duckdb:///:memory:'

country,po_count,total_spend,avg_rating
Germany,42,995411.84,4.2
USA,32,819898.26,4.5
UK,29,752094.86,3.8
